In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

# 1. الأماكن
csv_file_path = 'egypt_traffic_points.csv' 
try:
    df_points = pd.read_csv(csv_file_path)
    locations = df_points['LocationName'].dropna().tolist()
except Exception as e:
    print(f"⚠️ لم يتم العثور على الملف، سيتم استخدام أماكن افتراضية.")
    locations = [
        "Stanley Bridge", "Corniche - San Stefano", "Alexandria Agriculture Road",
        "Ain Shams University Area", "Cairo Ring Road", "Tahrir Square", "Oasis Road"
    ]

# 2. حالات السائق
adas_states = [
    'SAFE & FOCUSED', 
    'SOS: FAINTING / EMERGENCY!', 
    'DISTRACTED: PHONE USAGE!', 
    'DROWSY - SLEEPING!', 
    'STRESS / ANGER DETECTED!', 
    'DISTRACTED: LOOKING AWAY!', 
    'FATIGUE - YAWNING!'
]

start_date = datetime.now() - timedelta(days=180)
incidents_data = []

print("⏳ جاري توليد 10,000 سجل حوادث متكامل (ADAS + طقس + مرور)...")

# 3. حلقة توليد 10,000 حادثة
for i in range(1, 10001):
    loc = random.choice(locations)
    
    # تحديد وقت عشوائي
    random_days = random.randint(0, 180)
    random_hours = random.randint(0, 23)
    random_minutes = random.randint(0, 59)
    incident_datetime = start_date + timedelta(days=random_days, hours=random_hours, minutes=random_minutes)
    
    # تحديد حالة السائق
    driver_status = random.choice(adas_states)
    
    # منطق سبب الحادثة
    if driver_status == 'SOS: FAINTING / EMERGENCY!':
        inc_type, cause, severity = 'Accident', 'Fainting / Sudden Illness', random.choice(['Major', 'Fatal'])
    elif driver_status == 'DISTRACTED: PHONE USAGE!':
        inc_type, cause, severity = 'Accident', 'Phone Usage', random.choice(['Moderate', 'Major'])
    elif driver_status in ['DROWSY - SLEEPING!', 'FATIGUE - YAWNING!']:
        inc_type, cause, severity = 'Accident', 'Driver Fatigue', random.choice(['Moderate', 'Major', 'Fatal'])
    elif driver_status == 'STRESS / ANGER DETECTED!':
        inc_type, cause, severity = 'Accident', random.choice(['Speeding', 'Sudden Braking']), random.choice(['Minor', 'Moderate'])
    elif driver_status == 'DISTRACTED: LOOKING AWAY!':
        inc_type, cause, severity = 'Accident', 'Lack of Attention', random.choice(['Minor', 'Moderate'])
    else: # SAFE & FOCUSED
        inc_type = random.choice(['Vehicle Breakdown', 'Roadworks', 'Weather Condition', 'Accident (Other Driver)'])
        cause = random.choice(['Engine Failure', 'Slippery Road', 'Unknown'])
        severity = random.choice(['Minor', 'Moderate'])

    # حساب وقت التأخير
    if severity == 'Minor': delay = random.randint(15, 30)
    elif severity == 'Moderate': delay = random.randint(30, 60)
    elif severity == 'Major': delay = random.randint(60, 120)
    else: delay = random.randint(120, 240)
        
    # منطق الطقس المرتبط بالحادثة
    if cause == 'Slippery Road' or inc_type == 'Weather Condition':
        weather_cond = random.choice(['Rainy', 'Foggy'])
    else:
        weather_cond = random.choice(['Clear Sky', 'Cloudy', 'Clear Sky', 'Clear Sky']) # فرصة أكبر للجو الصافي

    if weather_cond == 'Rainy':
        temp = round(random.uniform(15.0, 22.0), 1)
        humidity = random.randint(75, 95)
        wind = round(random.uniform(5.0, 15.0), 1)
    elif weather_cond == 'Foggy':
        temp = round(random.uniform(12.0, 19.0), 1)
        humidity = random.randint(85, 100)
        wind = round(random.uniform(0.0, 3.0), 1)
    else: # Clear or Cloudy
        temp = round(random.uniform(20.0, 38.0), 1)
        humidity = random.randint(30, 65)
        wind = round(random.uniform(2.0, 8.0), 1)

    # منطق المرور المرتبط بخطورة الحادثة
    if severity in ['Major', 'Fatal']:
        traffic_status = 'Heavy Traffic'
        current_speed = random.randint(0, 15)
    elif severity == 'Moderate':
        traffic_status = 'Moderate Traffic'
        current_speed = random.randint(15, 35)
    else:
        traffic_status = random.choice(['Free Flow', 'Moderate Traffic'])
        current_speed = random.randint(40, 80) if traffic_status == 'Free Flow' else random.randint(20, 40)

    # تجميع البيانات بنفس ترتيب أعمدة الـ SQL
    incidents_data.append({
        'Incident_ID': f"INC_{100000 + i}",
        'Location_Name': loc,
        'Incident_Date': incident_datetime.strftime('%Y-%m-%d'),
        'Incident_Time': incident_datetime.strftime('%H:%M:%S'),
        'Incident_Type': inc_type,
        'Severity': severity,
        'Cause': cause,
        'Road_Delay_Minutes': delay,
        'ADAS_Driver_Status': driver_status,
        'Temperature_C': temp,
        'Humidity_Pct': humidity,
        'Wind_Speed_ms': wind,
        'Weather_Condition': weather_cond,
        'CurrentSpeed': current_speed,
        'TrafficStatus': traffic_status
    })

# 4. حفظ البيانات
df_incidents = pd.DataFrame(incidents_data)
output_file = 'Mock_Incidents_10k_Full.csv'
df_incidents.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✅ تم بنجاح! تم إنشاء ملف '{output_file}' ويحتوي على {len(df_incidents)} حادثة.")

⏳ جاري توليد 10,000 سجل حوادث متكامل (ADAS + طقس + مرور)...
✅ تم بنجاح! تم إنشاء ملف 'Mock_Incidents_10k_Full.csv' ويحتوي على 10000 حادثة.


In [2]:
import pandas as pd
import pyodbc

SERVER_NAME = r'DESKTOP-2A0QBS7\SQLEXPRESS'
DATABASE_NAME = 'EgyptTrafficDB'
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};Trusted_Connection=yes;'

# 1. تغيير اسم الملف للـ 10,000 حادثة المتكاملة
csv_file = 'Mock_Incidents_10k_Full.csv' 

try:
    df = pd.read_csv(csv_file)
    print(f"⏳ جاري رفع {len(df)} حادثة لقاعدة البيانات...")
    
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    
    # تفريغ الجدول القديم
    cursor.execute("TRUNCATE TABLE Historical_Incidents_ADAS")
    
    # 2. تحديث الكويري ليضم كل الأعمدة الجديدة (15 عمود)
    insert_query = """
        INSERT INTO Historical_Incidents_ADAS 
        (Incident_ID, Location_Name, Incident_Date, Incident_Time, Incident_Type, 
         Severity, Cause, Road_Delay_Minutes, ADAS_Driver_Status, 
         Temperature_C, Humidity_Pct, Wind_Speed_ms, Weather_Condition, CurrentSpeed, TrafficStatus)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    
    # 3. استخدام executemany للرفع السريع بدل iterrows
    # بنحول الداتا لـ List of Tuples عشان تتوافق مع pyodbc
    data_to_insert = [tuple(x) for x in df.to_numpy()]
    
    cursor.fast_executemany = True # تسريع الرفع (Fast Insert)
    cursor.executemany(insert_query, data_to_insert)
    
    conn.commit()
    cursor.close()
    conn.close()
    print("✅ تم رفع الـ 10,000 حادثة بكامل تفاصيل الطقس والمرور بنجاح!")

except Exception as e:
    print(f"❌ حصل خطأ أثناء الرفع: {e}")

⏳ جاري رفع 10000 حادثة لقاعدة البيانات...
✅ تم رفع الـ 10,000 حادثة بكامل تفاصيل الطقس والمرور بنجاح!
